### Workflow

Inspect quotient-BP representations and hard-decision trajectories on saved fixed-weight cases, then compare damping, the C++ implementation and selective freezing. These are exploratory support-level comparisons rather than an unbiased physical-noise benchmark.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/quotient'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/quotient_bp_paths.json')


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

DATA_DIR = _data_paths.run()
DATA_DIR.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD, CythonBPOSD, QuotientBP
from src.decoders import Tesseract
from utils.trajectory_quotient_bp import *
from beliefmatching import detector_error_model_to_check_matrices


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
WEIGHT_DIR = _data_paths.input('error_weight')
os.makedirs(WEIGHT_DIR, exist_ok=True)
num_errs = 33
num_dets = 14
num_obs = 2
max_iter = 60
code = ca_codes['CA_9_2_5']


## Hard Decision Trajectory

### Select diagnostic inputs

Read the weight-2 table and select the recorded decoder-disagreement population. Reconstruct the chosen row's circuit and truth before comparing its trajectories.


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2_data = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=types)


In [ ]:
weight2_data


In [ ]:
mask = (weight2_data['pred_b'] != weight2_data['true']) & (weight2_data['pred_t'] == weight2_data['true'])
filtered_data = weight2_data[mask]
filtered_data


In [ ]:
filtered_data.shape


In [ ]:
row = filtered_data[(filtered_data['det'] == '11011000000110') & (filtered_data['err'] == '010000000100000000000000000000000')].iloc[-1]
error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
compiled_b = CythonBPOSD(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
compiled_t = Tesseract(code, error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)


In [ ]:
pred_b = compiled_b.decoder.decode(str2bits(row['det']))
belief_history = compiled_b.decoder.belief_history.copy()
pred_t = compiled_t.decoder.decode(str2bits(row['det']))
info_b = {
    'pred_b': pred_b,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_b.decoder.is_osd_satisfied,
}
info_t = {'pred_t': pred_t.astype(int)}

print('BPOSD info:', info_b)
print('Tesseract info:', info_t)


In [ ]:
hard_history = (belief_history < 0).astype(int)
L = compiled_b.decoder.L.toarray()


In [ ]:
hard_history @ L.T


### Construct the quotient representation

Build the quotient model from the parity-check and logical matrices, inspect its graph and try the selected correction. These diagnostic helpers make the representation visible independently of the later C++ comparison.


In [ ]:
from dataclasses import dataclass
from experiment.utils import bp_message_dynamics

Matrices = bp_message_dynamics.Matrices


In [ ]:
demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
H = demMatrices.check_matrix.toarray()
L = demMatrices.observables_matrix.toarray()
new_matrices = Matrices(H=H.T @ H % 2, L=L, p=demMatrices.priors)
new_dem = new_matrices.to_dem()


In [ ]:
model = build_quotient_model(H, L)
print('H shape:', H.shape)
print('L shape:', L.shape)
print('rank(H):', model.rank_H)
print('dim ker(H):', model.K.shape[1])
print('degeneracy dimension:', model.D.shape[1])
print('logical rank:', model.logical_rank)
print('Q shape:', model.Q.shape)
print('LQ =')
print(L @ model.Q % 2)


In [ ]:
quotient_graph_stats(model)


In [ ]:
e = str2bits('000000010000000000000000100010000')
syndrome = H @ e % 2
true_observable = L @ e % 2
result = decode_logical_coset_bp(model, syndrome, demMatrices.priors, h_max_iter=50, h_method='min_sum', d_max_iter=60)
correction = result['correction']
residual = e ^ correction

print('true observable:', true_observable)
print('chosen correction observable:', L @ correction % 2)
print('representative correction:', correction)
print('residual syndrome:', H @ residual % 2)
print('residual observable:', L @ residual % 2)


damping makes difference ??

### Evaluate the damping variant

Run the quotient trajectory on the selected support population with the stated damping setting. Compare with the no-damping block below using each block's actual configuration and denominator.


In [ ]:
fails = 0
total = weight2_data.shape[0]
for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = H @ e % 2
    true_observable = L @ e % 2
    result = decode_logical_coset_bp(model, syndrome, demMatrices.priors, h_max_iter=50, h_method='min_sum', d_max_iter=60)
    correction = result['correction']
    residual = e ^ correction
    correction_observable = L @ correction % 2

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


no damping

In [ ]:
fails = 0
total = weight2_data.shape[0]
for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = H @ e % 2
    true_observable = L @ e % 2
    result = decode_logical_coset_bp(
        model,
        syndrome,
        demMatrices.priors,
        h_damping=0,
        h_max_iter=50,
        h_method='min_sum',
        d_max_iter=60,
    )
    correction = result['correction']
    residual = e ^ correction
    correction_observable = L @ correction % 2

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


### Extend the comparison to weight 3

Read the weight-3 cases and inspect different decoder-outcome subsets. Changing the outcome mask changes the selected population, so keep the cohort label with every failure count.


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight3_data = pd.read_csv(_data_paths.input('weight_3_data_csv'), dtype=types)


In [ ]:
mask = (weight3_data['pred_b'] != weight3_data['true']) & (weight3_data['pred_t'] == weight3_data['true'])
filtered_data = weight3_data[mask]
filtered_data


In [ ]:
filtered_data['true'].value_counts()


In [ ]:
fails = 0
total = filtered_data.shape[0]
for _, row in filtered_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = H @ e % 2
    true_observable = L @ e % 2
    result = decode_logical_coset_bp(model, syndrome, demMatrices.priors, h_max_iter=50, h_method='min_sum', d_max_iter=60)
    correction = result['correction']
    residual = e ^ correction
    correction_observable = L @ correction % 2

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


In [ ]:
mask = (weight3_data['pred_b'] == weight3_data['true']) & (weight3_data['pred_t'] != weight3_data['true'])
filtered_data = weight3_data[mask]
filtered_data


In [ ]:
filtered_data['true'].value_counts()


In [ ]:
fails = 0
total = filtered_data.shape[0]
for _, row in filtered_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = H @ e % 2
    true_observable = L @ e % 2
    result = decode_logical_coset_bp(model, syndrome, demMatrices.priors, h_max_iter=50, h_method='min_sum', d_max_iter=60)
    correction = result['correction']
    residual = e ^ correction
    correction_observable = L @ correction % 2

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


In [ ]:
mask = (weight3_data['pred_b'] != weight3_data['true']) & (weight3_data['pred_t'] != weight3_data['true'])
filtered_data = weight3_data[mask]
filtered_data


In [ ]:
filtered_data['true'].value_counts()


In [ ]:
mask = (weight3_data['pred_b'] == weight3_data['true']) & (weight3_data['pred_t'] == weight3_data['true'])
filtered_data = weight3_data[mask]
filtered_data


In [ ]:
filtered_data['true'].value_counts()


### CPP 

### Compare the C++ decoder path

Run the existing compiled QuotientBP implementation on the loaded cases. This is a backend experiment with its own settings, not a claim of bitwise equivalence to the diagnostic trajectory helpers.


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2_data = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=types)


In [ ]:
fails = 0
total = weight2_data.shape[0]
for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    compiled_b = QuotientBP(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
    correction = compiled_b.decoder.decode_to_correction(str2bits(row['det']))
    correction_observable = compiled_b.decoder.L @ correction % 2
    true_observable = str2bits(row['true'])

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


## Frozen

### Test selective freezing

Evaluate the frozen-variable quotient variant under its explicit iteration, damping and unfrozen-variable settings. Interpret this selected-case comparison separately from the earlier damping runs.


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2_data = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=types)


In [ ]:
fails = 0
total = weight2_data.shape[0]
for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = H @ e % 2
    true_observable = L @ e % 2
    result = decode_reliability_frozen_quotient_bp(
        model,
        syndrome,
        demMatrices.priors,
        h_max_iter=50,
        h_method='min_sum',
        h_damping=0,
        d_max_iter=60,
        d_damping=0,
        max_unfrozen_variables=20,
    )
    correction = result['correction']
    residual = e ^ correction
    correction_observable = L @ correction % 2

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')
